# Fine-Tune an AI Friend with Qwen3 0.6B and LoRA

This notebook fine-tunes `Qwen/Qwen3-0.6B` with LoRA supervised fine-tuning on a Telegram `result.json` export.

The task is: given recent chat context, generate the selected friend's next message.

Expected outputs:

- A trained LoRA adapter saved to `qwen3-0.6b-ai-friend-lora/`
- Safe aggregate dataset statistics
- Manual generation examples
- A small Gradio chat demo at the end

## Privacy Warning

Telegram exports can contain highly private information. Do not publish your `result.json`, generated training examples, trained adapter, logs, screenshots, or generated outputs unless every participant has consented and you have reviewed the content carefully.

This notebook is designed to avoid printing raw Telegram messages by default. It only prints aggregate statistics and redacted example structure.


## 1. Runtime Setup

This notebook targets Google Colab with a GPU runtime. Local execution may work for parsing and dataset preparation, but full training is intended for Colab GPU.


In [ ]:
%pip install -q -U transformers datasets peft trl accelerate bitsandbytes gradio sentencepiece


## 2. Imports and Configuration

Adjust the configuration if you need shorter context, a different output directory, or smaller training settings for a limited GPU.


In [ ]:
import json
import random
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path

MODEL_NAME = "Qwen/Qwen3-0.6B"
DATA_PATH = "result.json"
OUTPUT_DIR = "qwen3-0.6b-ai-friend-lora"

MAX_CONTEXT_MESSAGES = 8
MAX_SEQ_LENGTH = 1024
TRAIN_TEST_SPLIT = 0.05
SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

NUM_TRAIN_EPOCHS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = "cosine"
LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 100
MAX_GRAD_NORM = 0.3

random.seed(SEED)


## 3. Load Telegram Export

Upload `result.json` to the root of the Colab working directory before running this section.

The parser handles Telegram text fields that are strings, lists of strings, or lists of text entity dictionaries. It ignores service records and messages without usable text.


In [ ]:
SPECIAL_MARKERS = ("<|system|>", "<|context|>", "<|answer|>")


def telegram_text_to_plain(value):
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(telegram_text_to_plain(item.get("text", "")))
        return "".join(parts)
    return ""


def clean_message_text(text):
    text = str(text).replace("\u00a0", " ")
    for marker in SPECIAL_MARKERS:
        text = text.replace(marker, "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_sender_name(value):
    if value is None:
        return "Unknown"
    value = str(value).strip()
    return value or "Unknown"


def make_speaker_key(sender_name, sender_id):
    if sender_id is not None and str(sender_id).strip():
        return f"id:{sender_id}"
    return f"name:{sender_name}"


def parse_telegram_export(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Upload Telegram result.json to the current working directory first."
        )

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        raw_messages = data.get("messages", [])
    elif isinstance(data, list):
        raw_messages = data
    else:
        raw_messages = []

    parsed = []
    for original_index, record in enumerate(raw_messages):
        if not isinstance(record, dict):
            continue
        if record.get("type") != "message":
            continue

        text = clean_message_text(telegram_text_to_plain(record.get("text", "")))
        if not text:
            continue

        sender_name = normalize_sender_name(
            record.get("from") or record.get("actor") or record.get("author")
        )
        sender_id = record.get("from_id") or record.get("actor_id") or record.get("author_id")
        speaker_key = make_speaker_key(sender_name, sender_id)

        try:
            timestamp = int(record.get("date_unixtime") or 0)
        except (TypeError, ValueError):
            timestamp = 0

        message_id = record.get("id", original_index)
        try:
            message_id_for_sort = int(message_id)
        except (TypeError, ValueError):
            message_id_for_sort = original_index

        parsed.append(
            {
                "text": text,
                "sender_name": sender_name,
                "sender_id": sender_id,
                "speaker_key": speaker_key,
                "date": record.get("date", ""),
                "timestamp": timestamp,
                "message_id": message_id_for_sort,
                "original_index": original_index,
            }
        )

    parsed.sort(key=lambda item: (item["timestamp"], item["message_id"], item["original_index"]))
    return parsed


messages = parse_telegram_export(DATA_PATH)

if not messages:
    raise ValueError("No usable text messages were found in the Telegram export.")

speaker_count = len({message["speaker_key"] for message in messages})
first_date = next((message["date"] for message in messages if message.get("date")), "unknown")
last_date = next((message["date"] for message in reversed(messages) if message.get("date")), "unknown")

print(f"Parsed usable text messages: {len(messages):,}")
print(f"Detected speakers: {speaker_count:,}")
print(f"Date range: {first_date} to {last_date}")


## 4. Speaker Selection

Choose which speaker should be modeled as the friend. Only aggregate speaker statistics are shown here.

All non-selected speakers are treated as the conversation partner side and formatted as `User` in the training examples.


In [ ]:
def build_speaker_stats(parsed_messages):
    counts = Counter(message["speaker_key"] for message in parsed_messages)
    names = {}
    ids = {}

    for message in parsed_messages:
        key = message["speaker_key"]
        names.setdefault(key, message["sender_name"])
        ids.setdefault(key, message["sender_id"])

    total = sum(counts.values())
    rows = []
    for index, (speaker_key, count) in enumerate(counts.most_common()):
        rows.append(
            {
                "index": index,
                "speaker_key": speaker_key,
                "sender_name": names.get(speaker_key, "Unknown"),
                "sender_id": ids.get(speaker_key),
                "message_count": count,
                "percentage": 100.0 * count / total if total else 0.0,
            }
        )
    return rows


speaker_stats = build_speaker_stats(messages)

print(f"{'index':>5}  {'messages':>10}  {'percent':>8}  {'sender_id':<24}  sender_name")
for row in speaker_stats:
    sender_id = str(row["sender_id"] or "")[:24]
    print(
        f"{row['index']:>5}  {row['message_count']:>10,}  {row['percentage']:>7.2f}%  "
        f"{sender_id:<24}  {row['sender_name']}"
    )

selected_index = int(input("Enter the speaker index to model as the friend: ").strip())
if selected_index < 0 or selected_index >= len(speaker_stats):
    raise ValueError("Selected speaker index is out of range.")

selected_speaker = speaker_stats[selected_index]
TARGET_SPEAKER_KEY = selected_speaker["speaker_key"]
TARGET_SPEAKER_NAME = selected_speaker["sender_name"]
TARGET_SPEAKER_ID = selected_speaker["sender_id"]

print(f"Selected target speaker index: {selected_index}")
print(f"Selected target speaker name: {TARGET_SPEAKER_NAME}")
print(f"Selected target speaker id: {TARGET_SPEAKER_ID or 'not available'}")


## 5. Build Supervised Training Examples

For every message written by the selected friend, the notebook uses the previous `MAX_CONTEXT_MESSAGES` messages as context and trains the model to generate that friend's next message.

The custom text format is explicit and simple:

```text
<|system|>
You are imitating the writing style of a specific person in a private Telegram chat.
Given the recent chat context, write only this person's next message.
Do not explain yourself.
Do not add metadata.
<|context|>
Friend: [message hidden]
User: [message hidden]
<|answer|>
[message hidden]
```

Version 1 uses standard causal language modeling over the full prompt and answer. This is simpler and more compatible across TRL versions than answer-only masking.


In [ ]:
SYSTEM_PROMPT = """<|system|>
You are imitating the writing style of a specific person in a private Telegram chat.
Given the recent chat context, write only this person's next message.
Do not explain yourself.
Do not add metadata."""


def role_for_message(message):
    return "Friend" if message["speaker_key"] == TARGET_SPEAKER_KEY else "User"


def format_training_example(context_messages, answer_text):
    context_lines = [f"{role_for_message(message)}: {message['text']}" for message in context_messages]
    return f"{SYSTEM_PROMPT}\n<|context|>\n" + "\n".join(context_lines) + f"\n<|answer|>\n{answer_text}"


def build_training_texts(parsed_messages, max_context_messages):
    texts = []
    shapes = []

    for index, message in enumerate(parsed_messages):
        if message["speaker_key"] != TARGET_SPEAKER_KEY:
            continue

        context_messages = parsed_messages[max(0, index - max_context_messages):index]
        if not context_messages:
            continue

        answer_text = message["text"]
        texts.append(format_training_example(context_messages, answer_text))
        shapes.append(
            {
                "context_roles": [role_for_message(context_message) for context_message in context_messages],
                "context_message_count": len(context_messages),
                "answer_chars": len(answer_text),
                "example_chars": len(texts[-1]),
            }
        )

    return texts, shapes


training_texts, example_shapes = build_training_texts(messages, MAX_CONTEXT_MESSAGES)

if not training_texts:
    raise ValueError(
        "No training examples were created. Choose a speaker with messages that have previous context."
    )

print(f"Created supervised examples: {len(training_texts):,}")


## 6. Dataset Preparation

This section creates a Hugging Face `Dataset`, shuffles it with a fixed seed, and splits it into train and validation sets.

It prints only safe aggregate statistics and a fully redacted example structure.


In [ ]:
from datasets import Dataset


def percentile(values, pct):
    if not values:
        return 0
    values = sorted(values)
    index = min(len(values) - 1, max(0, round((pct / 100) * (len(values) - 1))))
    return values[index]


def make_redacted_example(shape):
    lines = [f"{role}: [message hidden]" for role in shape["context_roles"]]
    return f"{SYSTEM_PROMPT}\n<|context|>\n" + "\n".join(lines) + "\n<|answer|>\n[message hidden]"


dataset = Dataset.from_dict({"text": training_texts}).shuffle(seed=SEED)

if len(dataset) > 1:
    validation_size = max(1, int(round(len(dataset) * TRAIN_TEST_SPLIT)))
    validation_size = min(validation_size, len(dataset) - 1)
    split_dataset = dataset.train_test_split(test_size=validation_size, seed=SEED)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
else:
    train_dataset = dataset
    eval_dataset = None

message_counts_by_speaker = Counter(message["speaker_key"] for message in messages)
example_char_lengths = [shape["example_chars"] for shape in example_shapes]
answer_char_lengths = [shape["answer_chars"] for shape in example_shapes]

print(f"Parsed messages: {len(messages):,}")
print(f"Speakers: {len(message_counts_by_speaker):,}")
print(f"Selected target speaker: index {selected_index}, messages {message_counts_by_speaker[TARGET_SPEAKER_KEY]:,}")
print(f"Training examples: {len(dataset):,}")
print(f"Train size: {len(train_dataset):,}")
print(f"Validation size: {len(eval_dataset) if eval_dataset is not None else 0:,}")
print(
    "Approx example character length: "
    f"p50={percentile(example_char_lengths, 50):,}, "
    f"p90={percentile(example_char_lengths, 90):,}, "
    f"max={max(example_char_lengths):,}"
)
print(
    "Approx answer character length: "
    f"p50={percentile(answer_char_lengths, 50):,}, "
    f"p90={percentile(answer_char_lengths, 90):,}, "
    f"max={max(answer_char_lengths):,}"
)

print("\nRedacted example structure:\n")
print(make_redacted_example(example_shapes[0]))


## 7. Load Tokenizer and Model

This section loads `Qwen/Qwen3-0.6B`. On GPU, it prefers 4-bit quantization with bitsandbytes and prepares the model for k-bit LoRA training.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def append_eos(batch):
    eos = tokenizer.eos_token
    if not eos:
        return batch
    return {"text": [text if text.endswith(eos) else text + eos for text in batch["text"]]}

train_dataset = train_dataset.map(append_eos, batched=True)
if eval_dataset is not None:
    eval_dataset = eval_dataset.map(append_eos, batched=True)

sample_for_lengths = train_dataset.select(range(min(len(train_dataset), 1000)))
token_lengths = [
    len(tokenizer(text, add_special_tokens=False)["input_ids"])
    for text in sample_for_lengths["text"]
]
print(
    "Approx token length from train sample: "
    f"p50={percentile(token_lengths, 50):,}, "
    f"p90={percentile(token_lengths, 90):,}, "
    f"max={max(token_lengths) if token_lengths else 0:,}"
)

bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16
USE_4BIT = torch.cuda.is_available()

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

model_load_kwargs = {"trust_remote_code": True}
if torch.cuda.is_available():
    model_load_kwargs.update({"device_map": "auto", "torch_dtype": compute_dtype})
    if quantization_config is not None:
        model_load_kwargs["quantization_config"] = quantization_config
else:
    model_load_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_load_kwargs)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

model.config.use_cache = False
print(f"Loaded {MODEL_NAME}")
print(f"4-bit training: {USE_4BIT}")
print(f"bf16 supported: {bf16_supported}")


## 8. Configure LoRA

The LoRA configuration targets common Qwen projection modules and keeps the adapter small enough for Colab experimentation.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 9. Fine-Tuning

This cell uses TRL `SFTTrainer` when available. If the installed TRL API is incompatible, it falls back to a standard Hugging Face `Trainer` for causal language modeling.

The defaults are conservative for Colab: small per-device batch size, gradient accumulation, and two epochs.


In [ ]:
import inspect
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments


def filtered_kwargs(cls, kwargs):
    params = inspect.signature(cls.__init__).parameters
    return {key: value for key, value in kwargs.items() if key in params}


def add_eval_strategy(kwargs, params, has_eval):
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps" if has_eval else "no"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps" if has_eval else "no"
    if has_eval:
        kwargs["eval_steps"] = EVAL_STEPS


def base_training_kwargs(arg_cls, has_eval):
    params = inspect.signature(arg_cls.__init__).parameters
    kwargs = {
        "output_dir": OUTPUT_DIR,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "logging_steps": LOGGING_STEPS,
        "save_steps": SAVE_STEPS,
        "save_strategy": "steps",
        "max_grad_norm": MAX_GRAD_NORM,
        "report_to": "none",
        "seed": SEED,
        "bf16": bf16_supported,
        "fp16": torch.cuda.is_available() and not bf16_supported,
    }
    if torch.cuda.is_available() and "optim" in params:
        kwargs["optim"] = "paged_adamw_8bit"
    add_eval_strategy(kwargs, params, has_eval)
    return filtered_kwargs(arg_cls, kwargs)


has_eval = eval_dataset is not None and len(eval_dataset) > 0

try:
    from trl import SFTConfig, SFTTrainer

    sft_params = inspect.signature(SFTConfig.__init__).parameters
    sft_kwargs = base_training_kwargs(SFTConfig, has_eval)
    if "max_seq_length" in sft_params:
        sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
    elif "max_length" in sft_params:
        sft_kwargs["max_length"] = MAX_SEQ_LENGTH
    if "dataset_text_field" in sft_params:
        sft_kwargs["dataset_text_field"] = "text"
    if "packing" in sft_params:
        sft_kwargs["packing"] = False

    training_args = SFTConfig(**sft_kwargs)

    trainer_params = inspect.signature(SFTTrainer.__init__).parameters
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
    }
    if has_eval:
        trainer_kwargs["eval_dataset"] = eval_dataset
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer
    if "dataset_text_field" in trainer_params:
        trainer_kwargs["dataset_text_field"] = "text"
    if "max_seq_length" in trainer_params:
        trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

    trainer = SFTTrainer(**trainer_kwargs)
    print("Using TRL SFTTrainer.")

except Exception as exc:
    print("SFTTrainer setup failed. Falling back to Hugging Face Trainer.")
    print(f"Reason: {type(exc).__name__}: {exc}")

    def tokenize_for_causal_lm(batch):
        tokenized = tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=False,
        )
        tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
        return tokenized

    tokenized_train = train_dataset.map(
        tokenize_for_causal_lm,
        batched=True,
        remove_columns=train_dataset.column_names,
    )
    tokenized_eval = None
    if has_eval:
        tokenized_eval = eval_dataset.map(
            tokenize_for_causal_lm,
            batched=True,
            remove_columns=eval_dataset.column_names,
        )

    training_args = TrainingArguments(**base_training_kwargs(TrainingArguments, has_eval))
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        data_collator=data_collator,
    )

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter and tokenizer files to: {OUTPUT_DIR}")


## 10. Save and Reload Adapter

The output directory contains the LoRA adapter configuration and weights, plus tokenizer files. Keep this directory private if the source chat is private.

This section reloads the base model and attaches the saved LoRA adapter for inference.


In [ ]:
import gc
from peft import PeftModel

try:
    del trainer
    del model
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_load_kwargs)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()
model.config.use_cache = True

print("Reloaded base model with LoRA adapter for inference.")


## 11. Generation Helper

The helper below formats recent context in the same style as training and generates only the friend's next message.

Context should be provided as role/message pairs, where role is `User` or `Friend`.


In [ ]:
def normalize_role(role):
    role = str(role).strip().lower()
    if role in {"friend", "assistant", "model"}:
        return "Friend"
    return "User"


def format_inference_prompt(context_messages):
    lines = []
    for role, text in context_messages[-MAX_CONTEXT_MESSAGES:]:
        clean_text = clean_message_text(text)
        if clean_text:
            lines.append(f"{normalize_role(role)}: {clean_text}")
    return f"{SYSTEM_PROMPT}\n<|context|>\n" + "\n".join(lines) + "\n<|answer|>\n"


def clean_generated_reply(text):
    text = text.replace(tokenizer.eos_token or "", "")
    for stop_marker in ["\nUser:", "\nFriend:", "<|context|>", "<|answer|>", "<|system|>"]:
        if stop_marker in text:
            text = text.split(stop_marker, 1)[0]
    return text.strip()


def generate_friend_reply(
    context_messages,
    max_new_tokens=80,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.05,
):
    prompt = format_inference_prompt(context_messages)
    device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return clean_generated_reply(generated_text)


## 12. Safe Generation Examples

Type your own short context below. The placeholder text is not taken from your Telegram export.

Do not paste sensitive text here if you plan to share the notebook output.


In [ ]:
manual_context = """
User: [write a recent message here]
Friend: [write a prior friend message here]
User: [write another message here]
""".strip()


def parse_manual_context(text):
    parsed_context = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if ":" in line:
            role, content = line.split(":", 1)
            parsed_context.append((normalize_role(role), content.strip()))
        else:
            parsed_context.append(("User", line))
    return parsed_context


if "[write" in manual_context:
    print("Edit manual_context with your own context, then rerun this cell.")
else:
    context = parse_manual_context(manual_context)
    for candidate_index in range(3):
        reply = generate_friend_reply(context)
        print(f"Candidate {candidate_index + 1}: {reply}")


## 13. Gradio Chat Demo

This demo runs inside the Colab session. It keeps recent chat history, formats it as context, and generates the friend's reply.

Quality depends on the size, consistency, and privacy sensitivity of your Telegram export.


In [ ]:
import gradio as gr


def gradio_reply_messages(message, history):
    context = []
    for item in history or []:
        role = "Friend" if item.get("role") == "assistant" else "User"
        context.append((role, item.get("content", "")))
    context.append(("User", message))
    return generate_friend_reply(context)


def gradio_reply_tuples(message, history):
    context = []
    for user_message, friend_message in history or []:
        if user_message:
            context.append(("User", user_message))
        if friend_message:
            context.append(("Friend", friend_message))
    context.append(("User", message))
    return generate_friend_reply(context)

try:
    demo = gr.ChatInterface(
        fn=gradio_reply_messages,
        type="messages",
        title="AI Friend LoRA Demo",
        description="Chat with the fine-tuned adapter in this Colab session. Keep private content private.",
    )
except TypeError:
    demo = gr.ChatInterface(
        fn=gradio_reply_tuples,
        title="AI Friend LoRA Demo",
        description="Chat with the fine-tuned adapter in this Colab session. Keep private content private.",
    )

demo.launch(share=True)
